#  Tầng 2b: Multiple Answer Cleanup

**Mục tiêu:** Nhận diện các cột Checkbox (nhiều đáp án phân cách bằng dấu phẩy),  
áp dụng One-Hot Encoding để tách thành các cột binary (0/1).

**Input:** `processed/cleaned_responses.csv`  
**Output:** `processed/multiple_answers_processed.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from pathlib import Path
# Ensure paths resolve from the project root when notebooks run from notebooks/.
import os
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
# ── Config ──────────────────────────────────────────────
INPUT_FILE = "processed/cleaned_responses.csv"
OUTPUT_FILE = "processed/multiple_answers_processed.csv"

assert Path(INPUT_FILE).exists(), f" Chưa có {INPUT_FILE}. Hãy chạy 1a_cleanup.ipynb trước!"
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
print(f" Đọc {INPUT_FILE}: {df.shape[0]} dòng × {df.shape[1]} cột")

 Đọc processed/cleaned_responses.csv: 157 dòng × 41 cột


In [2]:
# ══════════════════════════════════════════════════════════
# BƯỚC 1: TỰ ĐỘNG NHẬN DIỆN CỘT CHECKBOX
# ══════════════════════════════════════════════════════════
# Logic: Cột checkbox là cột object có ít nhất 1 giá trị chứa dấu phẩy
# và KHÔNG phải cột free-text (loại trừ Q35, Q36, Q37)

FREE_TEXT_COLS = ["Q35_improvement_suggestions", "Q36_university_suggestions", "Q37_skills_needed"]

def detect_checkbox_columns(dataframe, exclude_cols=None):
    """Tự động nhận diện cột checkbox dựa trên dấu phẩy trong giá trị."""
    exclude_cols = exclude_cols or []
    obj_cols = dataframe.select_dtypes(include="object").columns
    
    checkbox_cols = []
    for col in obj_cols:
        if col in exclude_cols or col == "timestamp":
            continue
        # Đếm số dòng có chứa dấu phẩy (loại bỏ NaN)
        has_comma = dataframe[col].dropna().str.contains(",", regex=False)
        comma_ratio = has_comma.mean()
        if comma_ratio > 0.1:  # > 10% dòng có dấu phẩy → checkbox
            checkbox_cols.append(col)
    
    return checkbox_cols

CHECKBOX_COLS = detect_checkbox_columns(df, exclude_cols=FREE_TEXT_COLS)

print(f" Phát hiện {len(CHECKBOX_COLS)} cột Checkbox:")
for col in CHECKBOX_COLS:
    n_multi = df[col].str.contains(",", na=False).sum()
    print(f"   • {col}  ({n_multi}/{len(df)} dòng có nhiều đáp án)")

 Phát hiện 8 cột Checkbox:
   • Q4_product_field  (105/157 dòng có nhiều đáp án)
   • Q7_tools_used  (19/157 dòng có nhiều đáp án)
   • Q9_usage_purpose  (42/157 dòng có nhiều đáp án)
   • Q10_cicd_benefits  (62/157 dòng có nhiều đáp án)
   • Q33_cicd_difficulties  (64/157 dòng có nhiều đáp án)
   • Q34_adoption_barriers  (75/157 dòng có nhiều đáp án)
   • Q39_biggest_barrier  (23/157 dòng có nhiều đáp án)
   • Q40_expected_benefit  (41/157 dòng có nhiều đáp án)


In [3]:
# ══════════════════════════════════════════════════════════
# BƯỚC 2: ONE-HOT ENCODING cho từng cột Checkbox
# ══════════════════════════════════════════════════════════

PREDEFINED_OPTIONS = {
    "Q4_product_field": [
        "Phần mềm ứng dụng (Web, Mobile, Desktop App)",
        "Hệ thống AI/Big Data/Data Science",
        "Hệ thống nhúng (Embedded)/IoT",
        "Giải pháp hạ tầng & Bảo mật"
    ],
    "Q7_tools_used": [
        "Jenkins",
        "GitLab CI/CD",
        "GitHub Actions",
        "Circle CI",
        "Travis CI",
        "Chưa từng sử dụng công cụ CI/CD nào"
    ],
    "Q8_learning_source": [
        "Môn học tại trường đại học",
        "Dự án cá nhân / Đồ án môn học",
        "Thực tập tại doanh nghiệp",
        "Khóa học online (Coursera, YouTube, ...)",
        "Tự tìm hiểu qua tài liệu trên Internet"
    ],
    "Q9_usage_purpose": [
        "Build / Tự động hoá quá trình build",
        "Continuous Integration (CI) – tích hợp & kiểm tra mã nguồn tự động",
        "Continuous Deployment / Delivery (CD) – triển khai tự động",
        "Automated Testing – kiểm thử tự động",
        "Tự động hoá quy trình phát triển (workflow automation)",
        "Học tập / nghiên cứu / thử nghiệm cá nhân"
    ],
    "Q10_cicd_benefits": [
        "Tự động hóa quy trình build và test",
        "Phát hiện lỗi sớm hơn",
        "Tăng tốc độ phát triển phần mềm",
        "Hỗ trợ làm việc nhóm tốt hơn",
        "Giảm lỗi khi triển khai phần mềm"
    ],
    "Q33_cicd_difficulties": [
        "Tích hợp liên tục (CI - Integration)",
        "Xây dựng (Build)",
        "Kiểm thử tự động (Automated Testing)",
        "Phân phối liên tục (CD - Delivery)",
        "Triển khai liên tục (CD - Deployment)",
        "Giám sát & Phản hồi"
    ],
    "Q34_adoption_barriers": [
        "Thiếu kiến thức hoặc kỹ năng về DevOps/CI/CD",
        "Thiếu tài liệu hoặc hướng dẫn thực hành rõ ràng",
        "Việc thiết lập và cấu hình công cụ CI/CD khá phức tạp",
        "Thiếu môi trường thực hành (server, cloud, Docker, v.v.)",
        "Ít cơ hội áp dụng CI/CD trong các môn học tại trường",
        "Chưa hiểu rõ lợi ích thực tế khi triển khai CI/CD"
    ],
    "Q39_biggest_barrier": [
        "Thiếu kiến thức hoặc kỹ năng về CI/CD và DevOps",
        "Khó thiết lập và cấu hình các công cụ CI/CD",
        "Thiếu môi trường thực hành (server, cloud, Docker, v.v.)",
        "Ít cơ hội thực hành CI/CD trong các môn học tại trường",
        "Dự án học tập còn đơn giản nên chưa thấy cần CI/CD",
        "Khó phối hợp làm việc nhóm khi áp dụng CI/CD"
    ],
    "Q40_expected_benefit": [
        "Giảm bớt các công việc thủ công lặp đi lặp lại (build, test, deploy)",
        "Phát hiện lỗi sớm hơn trong quá trình phát triển phần mềm.",
        "Tăng tốc độ phát triển và cập nhật dự án.",
        "Nâng cao kỹ năng DevOps và kinh nghiệm thực tế cho sinh viên."
    ]
}

def one_hot_encode_checkbox(series, col_prefix):
    predefined_list = PREDEFINED_OPTIONS.get(col_prefix, [])
    # Tạo DataFrame có giá trị 0 sẵn để gán
    dummies = pd.DataFrame(0, index=series.index, columns=[f"{col_prefix}__{opt}" for opt in predefined_list])
    
    for idx, val in series.items():
        if pd.isna(val) or val == "Không trả lời":
            continue
            
        # So khớp substring chính xác
        for opt in predefined_list:
            if opt in str(val):
                dummies.at[idx, f"{col_prefix}__{opt}"] = 1
                
        # Tìm và xử lý viết ý kiến tự do ("Mục khác" / custom write-ins)
        parts = [p.strip() for p in str(val).split(",") if p.strip()]
        for part in parts:
            # Kiểm tra xem đây có phải là một phần của đáp án chuẩn bị cắt sai hay không
            is_predefined = False
            for opt in predefined_list:
                if part in opt or opt in part:
                    is_predefined = True
                    break
            
            if not is_predefined:
                # Loại bỏ troll/test response rõ rệt
                if part.lower() in ["nhi béo", "nhi", "dsfdsfds", "ko", "không", "không có", "chưa có"]:
                    continue
                
                # Tạo cột mới động cho viết tự do
                col_name = f"{col_prefix}__{part}"
                if col_name not in dummies.columns:
                    dummies[col_name] = 0
                dummies.at[idx, col_name] = 1
                
    return dummies

print(" Hàm one_hot_encode_checkbox đã sẵn sàng")

 Hàm one_hot_encode_checkbox đã sẵn sàng


In [4]:
# ══════════════════════════════════════════════════════════
# BƯỚC 3: ÁP DỤNG ONE-HOT cho tất cả cột Checkbox
# ══════════════════════════════════════════════════════════

onehot_frames = []

for col in CHECKBOX_COLS:
    ohe = one_hot_encode_checkbox(df[col], col)
    onehot_frames.append(ohe)
    print(f"   {col} → {ohe.shape[1]} dummy columns")
    # Preview unique values
    short_names = [c.replace(f"{col}__", "")[:50] for c in ohe.columns]
    for name in short_names:
        print(f"      └─ {name}")
    print()

# Ghép tất cả dummy columns vào df gốc
df_processed = pd.concat([df] + onehot_frames, axis=1)

print(f"\n Shape sau One-Hot: {df_processed.shape[0]} dòng × {df_processed.shape[1]} cột")
print(f"   (Thêm {df_processed.shape[1] - df.shape[1]} cột dummy)")

   Q4_product_field → 8 dummy columns
      └─ Phần mềm ứng dụng (Web, Mobile, Desktop App)
      └─ Hệ thống AI/Big Data/Data Science
      └─ Hệ thống nhúng (Embedded)/IoT
      └─ Giải pháp hạ tầng & Bảo mật
      └─ Bay
      └─ Network
      └─ Game
      └─ Dữ liệu

   Q7_tools_used → 6 dummy columns
      └─ Jenkins
      └─ GitLab CI/CD
      └─ GitHub Actions
      └─ Circle CI
      └─ Travis CI
      └─ Chưa từng sử dụng công cụ CI/CD nào

   Q9_usage_purpose → 7 dummy columns
      └─ Build / Tự động hoá quá trình build
      └─ Continuous Integration (CI) – tích hợp & kiểm tra 
      └─ Continuous Deployment / Delivery (CD) – triển khai
      └─ Automated Testing – kiểm thử tự động
      └─ Tự động hoá quy trình phát triển (workflow automat
      └─ Học tập / nghiên cứu / thử nghiệm cá nhân
      └─ Chưa từng sử dụng

   Q10_cicd_benefits → 5 dummy columns
      └─ Tự động hóa quy trình build và test
      └─ Phát hiện lỗi sớm hơn
      └─ Tăng tốc độ phát triển phần mềm
 

In [5]:
# ══════════════════════════════════════════════════════════
# BƯỚC 4: THỐNG KÊ TẦN SUẤT cho mỗi cột Checkbox
# ══════════════════════════════════════════════════════════

for col in CHECKBOX_COLS:
    dummy_cols = [c for c in df_processed.columns if c.startswith(f"{col}__")]
    if dummy_cols:
        freq = df_processed[dummy_cols].sum().sort_values(ascending=False)
        freq.index = [c.replace(f"{col}__", "") for c in freq.index]
        print(f"\n {col} - Tần suất:")
        print(freq.to_string())
        print(f"   Total responses: {len(df_processed)}")


 Q4_product_field - Tần suất:
Phần mềm ứng dụng (Web, Mobile, Desktop App)    105
Hệ thống AI/Big Data/Data Science                32
Hệ thống nhúng (Embedded)/IoT                     6
Giải pháp hạ tầng & Bảo mật                       5
Bay                                               1
Network                                           1
Game                                              1
Dữ liệu                                           1
   Total responses: 157

 Q7_tools_used - Tần suất:
GitHub Actions                         49
Chưa từng sử dụng công cụ CI/CD nào    40
GitLab CI/CD                           24
Jenkins                                10
Circle CI                               7
Travis CI                               5
   Total responses: 157

 Q9_usage_purpose - Tần suất:
Học tập / nghiên cứu / thử nghiệm cá nhân                             56
Build / Tự động hoá quá trình build                                   44
Continuous Deployment / Delivery (CD) – triển kh

In [6]:
# ══════════════════════════════════════════════════════════
# BƯỚC 5: LƯU FILE
# ══════════════════════════════════════════════════════════

df_processed.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# Verify
verify = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
assert verify.shape == df_processed.shape
print(f" Đã lưu: {OUTPUT_FILE}")
print(f"   Shape: {verify.shape[0]} dòng × {verify.shape[1]} cột")
print(f"   Trong đó: {len(df.columns)} cột gốc + {verify.shape[1] - len(df.columns)} cột One-Hot")

 Đã lưu: processed/multiple_answers_processed.csv
   Shape: 157 dòng × 93 cột
   Trong đó: 41 cột gốc + 52 cột One-Hot
